In [5]:
import os, json
from datasets import load_dataset, Dataset, Features, Value

def normalize(ex):
    # add missing column
    if "difficulty" not in ex:
        ex["difficulty"] = None
    # ensure string columns
    for k in ("prompt", "task_name", "ability", "language", "meta"):
        if k in ex and not isinstance(ex[k], str):
            ex[k] = json.dumps(ex[k], ensure_ascii=False)
    # coerce answer to string
    ans = ex.get("answer", None)
    if isinstance(ans, list) or isinstance(ans, (dict, tuple)):
        ex["answer"] = json.dumps(ans, ensure_ascii=False)
    elif ans is None:
        ex["answer"] = ""
    else:
        ex["answer"] = str(ans)
    return ex

# 1) Stream to avoid strict schema checks
stream = load_dataset(
    "BytedTsinghua-SIA/Enigmata-Data",
    streaming=True,
    cache_dir=os.path.join(os.environ.get('HF_HOME', '.'), 'data', 'huggingface', 'enigmata'),
)

# 2) Normalize each split and materialize locally
features = Features({
    "prompt": Value("string"),
    "answer": Value("string"),
    "task_name": Value("string"),
    "ability": Value("string"),
    "language": Value("string"),
    "meta": Value("string"),
    "difficulty": Value("string"),
})
fixed = {}
for split in stream.keys():
    iterable = stream[split].map(normalize)
    fixed[split] = Dataset.from_generator(lambda it=iterable: (ex for ex in it), features=features)

# fixed["train"], fixed["validation"], fixed["test"] are regular Datasets

Generating train split: 85719 examples [00:27, 5085.32 examples/s] Failed to load JSON from file 'hf://datasets/BytedTsinghua-SIA/Enigmata-Data@95f6920c2428c762ba6a6a2038af7dbe15eb7c75/hamiltonian_cycle/en/train.jsonl' with error <class 'pyarrow.lib.ArrowInvalid'>: JSON parse error: Column(/answer) changed from string to array in row 1
Generating train split: 88616 examples [00:28, 3148.62 examples/s]


DatasetGenerationError: An error occurred while generating the dataset